# 02 | Building the search and AI pipeline

**RAG means retrieval-augmented generation:** I search the published incidents first, then give those results to an LLM as evidence. I haven't trained a new language model.

The important design choice is to keep **historical reports**, **checked technical instructions** and **AI suggestions** separate. The app and notebooks use the same functions from `src/`.

**How I organised the build:** these notebooks call reusable functions from `src/` rather than copying the whole app. The [code tour](../docs/code_tour.md) explains the TF-IDF matrix, scoring, prompt construction and case storage.

In [ ]:
# ROOT finds the repository whether I open this notebook from notebooks/ or the project root.
# sys.path lets me import the same functions that the app uses.
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.incident_assistant import (
    LOCAL_CSV, load_incidents, prepare_incidents,
    build_index, retrieve, keyword_search, build_messages,
)

records = None
index = None
if LOCAL_CSV.is_file():
    records = prepare_incidents(load_incidents(LOCAL_CSV))
    index = build_index(records)
    print(f"Indexed {len(index.records):,} source-linked incidents")
else:
    print("Download the public CSV before running the live search examples.")

## 1. Compare the two search methods

My baseline counts overlapping words. TF-IDF gives extra weight to distinctive words and **word pairs**. I return five results for each method, using the same input.

This is a development check; my recorded 12-question comparison is in Notebook 03.

In [ ]:
question = "Have there been API timeout incidents affecting service requests?"
plain = []
ranked = []
if index is not None:
    plain = keyword_search(question, records, k=5)
    ranked = retrieve(question, index, k=5)
    # Print the vendor and title for each result, so I can inspect the matches.
    for record in plain:
        print("Keyword:", record["vendor"], record["title"])
    for record in ranked:
        print("TF-IDF:", record["vendor"], record["title"])
else:
    print("No local CSV, so no new search results were generated.")

## 2. Build a grounded prompt

Each retrieved record has an ID and an original link. The prompt tells Llama to use those sources, cite them and say when the information is missing.

I also built **18 reviewed technical-reference cards** and **six checked playbooks**. They help with **what to investigate**, not with claiming a confirmed cause from a status-page title.

In [ ]:
# A chat prompt is a list of messages: instructions first, then the user question and evidence.
if ranked:
    prompt = build_messages(question, ranked)
    print(prompt[0]['content'][:800])
    print('Evidence IDs:', [r['doc_id'] for r in ranked])

## 3. Keep local Llama optional

The local model is Ollama Llama 3.2. It may be slow on a laptop, so this cell is **off by default**; it is not part of the measured 15-answer evaluation. The real working demo is in Streamlit.

I don't send confidential IT tickets to a hosted provider.

In [ ]:
import os
from src.incident_assistant import answer_with_llm

RUN_LOCAL_LLAMA = False
if RUN_LOCAL_LLAMA and ranked:
    response = answer_with_llm(
        question, ranked,
        api_key="ollama",
        model=os.getenv("OPS_LLM_MODEL", "llama3.2"),
        base_url=os.getenv("OPS_LLM_BASE_URL", "http://localhost:11434/v1"),
        max_tokens=200,
    )
    print(response["answer"])
    print("Seconds:", response["seconds"], "Tokens:", response["tokens"])
else:
    print("Llama skipped. Retrieval and checked guides work without the model.")

## 4. Select a checked first step

The guide selector uses what the person says about the issue. An incident about somebody else's PostgreSQL problem must not make Scout assume our system uses PostgreSQL.

Here I show a short example without calling any model.

In [ ]:
from src.scout_playbooks import (
    load_playbooks, select_playbooks, guide_led_answer,
)
example = "Our checkout API has 504 errors after last night's deployment."
guides = select_playbooks(example, load_playbooks())
print("Guides selected:")
for guide in guides:
    print(guide["title"])
print("Suggested plan:")
print(guide_led_answer(guides, example)[:700])

## 5. The weakness that changed my first design

The original app waited for Llama **before** showing its checked plan. In my CPU-only diagnostic the model generated about **7.1 tokens/second**; the original raw-answer run had a **median of 81 seconds**. I also saw an unsupported PostgreSQL suggestion.

I did not need Llama to make the user wait for a checked first step. I changed the flow, rather than claim the model itself had become faster:

1. **First:** retrieve source links, select checked guides from the **user's reported facts**, show the first sensible check and save the reply.
2. **If requested:** ask Llama to interpret the saved evidence, clearly label its draft **unverified**, and keep the original plan intact.
3. **Throughout:** record messages, sources, notes and human reviews in SQLite, so one incident can continue over several questions.

The app never executes the suggested commands.

## 6. My design decision

The baseline found **9/12** target reports, compared with **8/12** for TF-IDF. TF-IDF was much faster **after indexing** (0.007 vs 0.551 seconds per query), so I kept it as a practical prototype retriever. This is a **speed trade-off, not an accuracy win**.

The revised source-first interface has software tests, but I have **not yet formally measured its end-to-end UI time or answer quality**. The 15 recorded Llama answers test the **original raw generation path**, not the newer checked-plan workflow.

[Notebook 03](03_evaluation.ipynb) evaluates retrieval and Llama separately. [Notebook 04](04_case_memory.ipynb) shows how I stored cases; [Notebook 05](05_checked_guides.ipynb) explains why the checked guide and human-review safeguards were added.

## 7. My final workflow: don't ask the same question twice

During the demo, Scout kept recommending a timeline because the original symptoms were still in the conversation. The user had supplied clues, but the program did not know whether a specific check had been completed. I added an explicit result-recording action, separate from a general chat message or human review of an answer.

`save_investigation_turn()` saves the reported finding against its guide ID. I read those saved notes with `completed_checks()`, then filter the applicable guides. This is a simple rule, not a new machine-learning model. Here I show that rule with a fictional finding; no actual company system is checked.

In [ ]:
from src.scout_workflow import completed_checks

# This is the same note format the real app saves in SQLite.
my_notes = [
    {"kind": "confirmed_fact",
     "content": "Completed check PB-TIMELINE-01: Example release 09:00, first errors 09:15 UTC."}
]
finished = completed_checks(my_notes)
applicable = select_playbooks(example, load_playbooks())

# I keep the guides that still need a result. A normal loop makes
# the change in the investigation order easy to follow.
remaining = []
for guide in applicable:
    if guide["id"] not in finished:
        remaining.append(guide)

print("Completed:", list(finished))
print("Next check:", remaining[0]["title"])
assert remaining[0]["id"] == "PB-DEPENDENCIES-01"

**What this gives me:** the timeline stays in the case record, while the next recommendation is the shared-systems check. The final demo uses the real shared workflow for all five checks, a fictional recovery result and an explicitly recorded human review. Scout cannot independently verify that recovery or prove a root cause. See [case memory](04_case_memory.ipynb) and [the code tour](../docs/code_tour.md).